In [20]:
import json
import cv2
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from matplotlib import animation
from IPython.display import HTML, Video, display
import tqdm as tqdm
from skimage.feature import local_binary_pattern
from scipy.spatial.distance import cdist
from scipy.spatial.distance import pdist
import os

In [21]:
df = pd.read_csv("/home/guilherme_monteiro/projetos/tcc/data/metadata/video-metadata-publish-with-links.csv")
df['video_path'] = df['Filename'].apply(lambda x: "/home/guilherme_monteiro/projetos/tcc/data/videos/" + x)
df_true = df[df['Video Ground Truth'] == "Real"].reset_index(drop=True)
df_fake = df[df['Video Ground Truth'] == "Fake"].reset_index(drop=True)

# Funções básicas

In [22]:
def load_video_frames(video_path, max_frames=None):
    cap = cv2.VideoCapture(video_path)

    frames = []
    count = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frames.append(frame)

        count += 1
        if max_frames and count >= max_frames:
            break

    cap.release()

    return np.array(frames)

def standardize_frame(frame, max_size=640):
    h, w = frame.shape[:2]

    scale = min(max_size / max(h, w), 1.0)
    new_w, new_h = int(w * scale), int(h * scale)

    frame_resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

    return frame_resized, scale

def load_metadata(video_path):
    path = video_path.replace(".mp4", "_meta.json")

    full_path = os.path.join("/home/guilherme_monteiro/projetos/tcc/data/metadata", os.path.basename(path))

    with open(full_path, "r") as f:
        return json.load(f)

def create_face_regions(frame, bbox, padding=0.2):
    h, w = frame.shape[:2]

    x1, y1, x2, y2 = bbox

    bw = x2 - x1
    bh = y2 - y1

    # aplica padding
    px = int(bw * padding)
    py = int(bh * padding)

    x1p = max(0, x1 - px)
    y1p = max(0, y1 - py)
    x2p = min(w, x2 + px)
    y2p = min(h, y2 + py)

    # máscara face interna
    face_mask = np.zeros((h, w), dtype=np.uint8)
    face_mask[y1:y2, x1:x2] = 1

    # máscara expandida
    expanded_mask = np.zeros((h, w), dtype=np.uint8)
    expanded_mask[y1p:y2p, x1p:x2p] = 1

    # borda = expandida - interna
    border_mask = expanded_mask - face_mask

    # fundo = resto
    background_mask = 1 - expanded_mask

    return {
        "face": face_mask,
        "border": border_mask,
        "background": background_mask,
        "bbox_expanded": (x1p, y1p, x2p, y2p)
    }


## Função para visualização

In [23]:
def draw_text_block(img, text_lines, x=10, y=20, line_height=18):
    for i, line in enumerate(text_lines):
        cv2.putText(
            img,
            line,
            (x, y + i * line_height),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 255),
            1,
            cv2.LINE_AA
        )
    return img

# Desenha a caixa delimitadora no frame
def draw_face_box(frame, bbox, color=(0, 255, 0), thickness=2):
    x1, y1, x2, y2 = bbox
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    return frame


def overlay_regions(frame, regions):
    overlay = frame.copy()

    face_mask = regions["face"]
    border_mask = regions["border"]
    background_mask = regions["background"]

    overlay[face_mask == 1] = (0, 255, 0)       # verde
    overlay[border_mask == 1] = (0, 0, 255)     # vermelho
    overlay[background_mask == 1] = (255, 0, 0) # azul

    alpha = 0.3
    return cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)

# SIFT

In [24]:
def create_sift():
    return cv2.SIFT_create(nfeatures=500)

def compute_sift_region(gray, mask, sift):

    mask_uint8 = (mask * 255).astype(np.uint8)

    keypoints, descriptors = sift.detectAndCompute(gray, mask_uint8)

    area = np.sum(mask)

    if descriptors is None or len(keypoints) == 0:

        return {
            "kp_density": 0.0,
            "desc_entropy": 0.0,
            "desc_self_similarity": 1.0,
            "descriptors": None,
            "has_kp": 0
        }

    # densidade
    kp_density = len(keypoints) / (area + 1e-6)

    # entropia
    desc_flat = descriptors.flatten()
    hist, _ = np.histogram(desc_flat, bins=32, density=True)
    desc_entropy = -np.sum(hist * np.log(hist + 1e-6))

    # auto-similaridade
    if len(descriptors) > 1:
        dists = pdist(descriptors, metric="cosine")
        desc_self_sim = 1 - np.mean(dists)
    else:
        desc_self_sim = 1.0

    return {
        "kp_density": kp_density,
        "desc_entropy": desc_entropy,
        "desc_self_similarity": desc_self_sim,
        "descriptors": descriptors,
        "has_kp": 1
    }

In [25]:
def sift_region_differences(f1, f2, prefix):

    if f1 is None or f2 is None:
        return {}

    features = {
        f"{prefix}_kp_density_diff": abs(f1["kp_density"] - f2["kp_density"]),
        f"{prefix}_desc_entropy_diff": abs(f1["desc_entropy"] - f2["desc_entropy"]),
        f"{prefix}_kp_presence_diff": abs(f1["has_kp"] - f2["has_kp"]),
    }

    if f1["descriptors"] is not None and f2["descriptors"] is not None:
        dists = cdist(f1["descriptors"], f2["descriptors"], metric="cosine")
        features[f"{prefix}_desc_dist"] = np.mean(dists)
    else:
        features[f"{prefix}_desc_dist"] = 1.0  # máximo (sem correspondência)

    return features

In [26]:
def compute_sift_metrics(frame, bbox):

    frame_std, _ = standardize_frame(frame)
    gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)

    regions = create_face_regions(frame_std, bbox)

    sift = create_sift()

    face = compute_sift_region(gray, regions["face"], sift)
    border = compute_sift_region(gray, regions["border"], sift)
    bg = compute_sift_region(gray, regions["background"], sift)

    features = {}

    # diretas
    if face:
        features["sift_face_kp_density"] = face["kp_density"]
        features["sift_face_entropy"] = face["desc_entropy"]
        features["sift_face_self_sim"] = face["desc_self_similarity"]

    # derivadas 
    features.update(sift_region_differences(face, bg, "face_bg"))
    features.update(sift_region_differences(face, border, "face_border"))
    features.update(sift_region_differences(border, bg, "border_bg"))

    return features

## Video para debbug

In [ ]:
def sift_visual(gray, regions, sift):

    vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

    colors = {
        "face": (0, 255, 0),        # verde
        "border": (0, 0, 255),      # vermelho
        "background": (255, 0, 0)   # azul
    }

    for region_name in ["face", "border", "background"]:

        mask = regions[region_name]

        ys, xs = np.where(mask == 1)

        if len(xs) < 20:
            continue

        x1, x2 = xs.min(), xs.max()
        y1, y2 = ys.min(), ys.max()

        patch = gray[y1:y2, x1:x2]

        kp, _ = sift.detectAndCompute(patch, None)

        for k in kp:
            px, py = int(k.pt[0]) + x1, int(k.pt[1]) + y1
            cv2.circle(vis, (px, py), 1, colors[region_name], -1)

    return vis

def debug_sift_video(video_path, max_frames=500, interval=60):

    frames = load_video_frames(video_path)

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, int(max_frames)).astype(int)
        frames = frames[idx]

    metadata = load_metadata(video_path)

    debug_frames = []

    sift = create_sift()

    print("\nProcessando SIFT...")

    for i, frame in enumerate(tqdm.tqdm(frames)):

        frame_std, scale = standardize_frame(frame)
        gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)

        bbox = metadata[i]["bbox"]
        bbox_scaled = [int(x * scale) for x in bbox]

        regions = create_face_regions(frame_std, bbox_scaled)

        # métricas
        features = compute_sift_metrics(frame, bbox_scaled)

        # visual SIFT
        sift_map = sift_visual(gray, regions, sift)

        # overlay regiões
        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {i}",

            f"Face KP Density: {features.get('sift_face_kp_density', 0):.4f}",
            f"Face Entropy: {features.get('sift_face_entropy', 0):.3f}",
            f"Face SelfSim: {features.get('sift_face_self_sim', 0):.3f}",

            f"Face-BG DescDist: {features.get('face_bg_desc_dist', 0):.3f}",
            f"Face-Border DescDist: {features.get('face_border_desc_dist', 0):.3f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(sift_map, cv2.COLOR_BGR2RGB)
        ])

        debug_frames.append(combined)

    print("\nRenderizando...")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")

    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(debug_frames),
        interval=interval,
        blit=True,
    )

    plt.close(fig)
    display(HTML(ani.to_jshtml()))

In [28]:
# debug_sift_video(df_true['video_path'].iloc[0], max_frames=300)

In [29]:
# debug_sift_video(df_fake['video_path'].iloc[0], max_frames=300)

# Patch Similarity

In [30]:
def extract_patches(gray, mask, patch_size=16, stride=8):

    patches = []

    h, w = gray.shape

    for y in range(0, h - patch_size, stride):
        for x in range(0, w - patch_size, stride):

            patch_mask = mask[y:y+patch_size, x:x+patch_size]

            # só considera patch com conteúdo suficiente da região
            #if np.mean(patch_mask) < 0.6:
            #    continue

            patch = gray[y:y+patch_size, x:x+patch_size]
            patches.append(patch.flatten())

    #if len(patches) < 10:
    #   return None

    return np.array(patches)

In [31]:
def compute_patch_similarity(patches):

    if patches is None:
        return None

    # distância entre patches
    dists = pdist(patches, metric="cosine")

    sims = 1 - dists

    return {
        "sim_mean": np.mean(sims),
        "sim_std": np.std(sims)
    }

def compute_patch_region(gray, mask):

    patches = extract_patches(gray, mask)

    if patches is None:
        return None

    return compute_patch_similarity(patches)

def patch_region_differences(f1, f2, prefix):

    if f1 is None or f2 is None:
        return {}

    return {
        f"{prefix}_sim_mean_diff": abs(f1["sim_mean"] - f2["sim_mean"]),
        f"{prefix}_sim_std_diff": abs(f1["sim_std"] - f2["sim_std"]),
    }

In [32]:
def compute_patch_metrics(frame, bbox):

    frame_std, _ = standardize_frame(frame)
    gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)

    regions = create_face_regions(frame_std, bbox)

    face = compute_patch_region(gray, regions["face"])
    border = compute_patch_region(gray, regions["border"])
    bg = compute_patch_region(gray, regions["background"])

    features = {}

    # diretas
    if face:
        features["patch_face_sim_mean"] = face["sim_mean"]
        features["patch_face_sim_std"] = face["sim_std"]

    # derivadas
    features.update(patch_region_differences(face, bg, "face_bg"))
    features.update(patch_region_differences(face, border, "face_border"))
    features.update(patch_region_differences(border, bg, "border_bg"))

    return features

## Video para debbug

In [ ]:
def patch_similarity_visual(gray, mask, patch_size=16, stride=8):

    vis = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

    h, w = gray.shape

    for y in range(0, h - patch_size, stride):
        for x in range(0, w - patch_size, stride):

            patch_mask = mask[y:y+patch_size, x:x+patch_size]

            if np.mean(patch_mask) < 0.6:
                continue

            patch = gray[y:y+patch_size, x:x+patch_size]

            var = np.var(patch)

            color = int(np.clip(var / 50, 0, 255))

            cv2.rectangle(vis, (x, y), (x+patch_size, y+patch_size),
                          (color, 255-color, 0), 1)

    return vis

def debug_patch_video(video_path, max_frames=500, interval=60):

    frames = load_video_frames(video_path)

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, int(max_frames)).astype(int)
        frames = frames[idx]

    metadata = load_metadata(video_path)

    debug_frames = []

    print("\nProcessando Patch Similarity...")

    for i, frame in enumerate(tqdm.tqdm(frames)):

        frame_std, scale = standardize_frame(frame)
        gray = cv2.cvtColor(frame_std, cv2.COLOR_BGR2GRAY)

        bbox = metadata[i]["bbox"]
        bbox_scaled = [int(x * scale) for x in bbox]

        regions = create_face_regions(frame_std, bbox_scaled)

        features = compute_patch_metrics(frame, bbox_scaled)

        patch_vis = patch_similarity_visual(gray, regions["face"])

        overlay = overlay_regions(frame_std.copy(), regions)
        overlay = draw_face_box(overlay, bbox_scaled)

        text_lines = [
            f"Frame: {i}",
            f"Face Sim Mean: {features.get('patch_face_sim_mean', 0):.3f}",
            f"Face Sim Std: {features.get('patch_face_sim_std', 0):.3f}",
            f"Face-BG SimDiff: {features.get('face_border_sim_mean_diff', 0):.3f}",
        ]

        overlay = draw_text_block(overlay, text_lines)

        combined = np.hstack([
            cv2.cvtColor(frame_std, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(patch_vis, cv2.COLOR_BGR2RGB)
        ])

        debug_frames.append(combined)

    print("\nRenderizando...")

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axis("off")

    img = ax.imshow(debug_frames[0])

    def update(i):
        img.set_data(debug_frames[i])
        return [img]

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(debug_frames),
        interval=interval,
        blit=True,
    )

    plt.close(fig)
    display(HTML(ani.to_jshtml()))

> Testar metodo sem patch similarity, explorar a metrica mais a fundo no futuro

## Todas as metricas

In [ ]:
def all_metrics(video_path, max_frames=500, label=None):
    video_name = os.path.basename(video_path)
    print(f"\nExtraindo métricas do vídeo: {video_name}")

    frames = load_video_frames(video_path)

    if len(frames) > max_frames:
        idx = np.linspace(0, len(frames) - 1, int(max_frames)).astype(int)
        frames = frames[idx]

    metadata = load_metadata(video_path)

    all_features = []

    for i, frame in enumerate(frames):

        frame_std, scale = standardize_frame(frame)

        bbox = metadata[i]["bbox"]
        bbox_scaled = [int(x * scale) for x in bbox]

        features_sift, _ = compute_sift_metrics(frame, bbox_scaled)

        features = {**features_sift,}
        features["frame"] = i

        all_features.append(features)

    df = pd.DataFrame(all_features)
    df.insert(0, "video_name", video_name)
    if label is not None:
        df.insert(1, "label", label)

    return df